# Notebook3 - SPC for AD

將 Shewhart、EWMA、CUSUM 套用在網路監控特徵上，觀察不同異常型態的敏感度。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
event_catalog = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])
display(features[["timestamp", "port_id", "traffic_total", "discard_rate", "error_rate", "event_label"]].head())

## Step 1 - 以正常期間估計 control limits

In [ ]:
rows = []
for port_id, g in features.groupby("port_id", sort=False):
    g = g.sort_values("timestamp").copy()
    base = g[g["event_label"] == "normal"].head(7 * 24 * 12)
    traffic_mu = base["traffic_total"].mean()
    traffic_sigma = base["traffic_total"].std()
    discard_mu = base["discard_rate"].mean()
    discard_sigma = base["discard_rate"].std()
    error_mu = base["error_rate"].mean()
    error_sigma = base["error_rate"].std()

    g["traffic_center"] = traffic_mu
    g["traffic_ucl"] = traffic_mu + 3 * traffic_sigma
    g["traffic_lcl"] = max(0, traffic_mu - 3 * traffic_sigma)
    g["shewhart_traffic_violation"] = g["traffic_total"] > g["traffic_ucl"]

    lam = 0.25
    g["ewma_discard_rate"] = g["discard_rate"].ewm(alpha=lam, adjust=False).mean()
    g["ewma_discard_ucl"] = discard_mu + 3 * discard_sigma * np.sqrt(lam / (2 - lam))
    g["ewma_discard_violation"] = g["ewma_discard_rate"] > g["ewma_discard_ucl"]

    k = 0.5 * max(error_sigma, 1e-12)
    h = 5 * max(error_sigma, 1e-12)
    s = 0.0
    pos = []
    for value in g["error_rate"].fillna(0):
        s = max(0, s + value - error_mu - k)
        pos.append(s)
    g["cusum_error_pos"] = pos
    g["cusum_error_h"] = h
    g["cusum_error_violation"] = g["cusum_error_pos"] > h
    rows.append(g)

spc = pd.concat(rows, ignore_index=True)
spc["any_spc_violation"] = spc[["shewhart_traffic_violation", "ewma_discard_violation", "cusum_error_violation"]].any(axis=1)
display(spc.loc[spc["any_spc_violation"], ["timestamp", "port_id", "event_label", "shewhart_traffic_violation", "ewma_discard_violation", "cusum_error_violation"]].head(12))

## Step 2 - 視覺化：Shewhart control chart

In [ ]:
port_id = "port-id7427"
p = spc[spc["port_id"] == port_id].copy()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(p["timestamp"], p["traffic_total"], label="traffic_total", linewidth=1)
ax.plot(p["timestamp"], p["traffic_center"], label="center", linewidth=1.1)
ax.plot(p["timestamp"], p["traffic_ucl"], label="UCL", color="tab:red", linewidth=1.1)
ax.plot(p["timestamp"], p["traffic_lcl"], label="LCL", color="tab:green", linewidth=1.1)
v = p[p["shewhart_traffic_violation"]]
ax.scatter(v["timestamp"], v["traffic_total"], color="tab:red", s=18, label="violation")
for _, event in event_catalog.iterrows():
    if event["port_id"] in [port_id, "MULTI"]:
        ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
ax.set_title(f"{port_id} - Shewhart chart for traffic_total")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 3 - 視覺化：EWMA 與 CUSUM

EWMA 適合小幅持續偏移，CUSUM 適合累積性變化。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(p["timestamp"], p["discard_rate"], alpha=0.35, label="discard_rate")
axes[0].plot(p["timestamp"], p["ewma_discard_rate"], label="EWMA discard_rate")
axes[0].plot(p["timestamp"], p["ewma_discard_ucl"], label="EWMA UCL", color="tab:red")
axes[0].set_title("EWMA chart for discard_rate")
axes[0].legend(loc="upper left")

axes[1].plot(p["timestamp"], p["cusum_error_pos"], label="CUSUM error positive")
axes[1].plot(p["timestamp"], p["cusum_error_h"], label="decision interval h", color="tab:red")
axes[1].set_title("CUSUM chart for error_rate")
axes[1].legend(loc="upper left")

for ax in axes:
    for _, event in event_catalog.iterrows():
        if event["port_id"] in [port_id, "MULTI"]:
            ax.axvspan(event["start_time"], event["end_time"], alpha=0.10, color="tab:red")
    ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 4 - SPC violation summary 與輸出

In [ ]:
summary = (
    spc.groupby("event_label")
    .agg(rows=("timestamp", "size"), spc_violations=("any_spc_violation", "sum"))
    .assign(violation_rate=lambda x: x["spc_violations"] / x["rows"])
    .sort_values("violation_rate", ascending=False)
)
display(summary)

keep = [
    "timestamp", "device_id", "port_id", "port_role", "event_label", "event_id",
    "traffic_total", "traffic_center", "traffic_ucl", "traffic_lcl", "shewhart_traffic_violation",
    "discard_rate", "ewma_discard_rate", "ewma_discard_ucl", "ewma_discard_violation",
    "error_rate", "cusum_error_pos", "cusum_error_h", "cusum_error_violation", "any_spc_violation",
]
out_path = DATA_PROCESSED / "spc_results.csv"
spc[keep].to_csv(out_path, index=False)
print(f"saved: {out_path}")